# IOAI — 2024 Final Stage Self Supervised Learning (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/val_x.npy'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-final-stage-self-supervised-learning/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 자기지도학습 — 모범답안 (SimCLR 대조학습)

미라벨 대량 데이터에 **SimCLR 대조학습**(지터·스케일·시간이동 증강 → NT-Xent)으로 인코더를 사전학습한 뒤, 동결하고 소수 라벨로 선형 프로브를 학습한다. 표현이 좋아져 val accuracy 가 크게 오른다(≈0.88 → 약 89점).

## 데이터·인코더·프로브 정의

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, pandas as pd
torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)
Xbig = torch.tensor(np.load("data/train_x_big.npy"), dtype=torch.float32)          # (N,3,206) 미라벨
Xs   = torch.tensor(np.load("data/train_x_small.npy"), dtype=torch.float32); ys = torch.tensor(np.load("data/train_y_small.npy"))
Xv   = torch.tensor(np.load("data/val_x.npy"), dtype=torch.float32)
mu = Xbig.mean((0,2), keepdim=True); sd = Xbig.std((0,2), keepdim=True) + 1e-6      # 채널 정규화(big 통계)
Xbig, Xs, Xv = (Xbig-mu)/sd, (Xs-mu)/sd, (Xv-mu)/sd
print("big", Xbig.shape, "small", Xs.shape, "val", Xv.shape)
class Encoder(nn.Module):
    def __init__(s):
        super().__init__(); s.f = nn.Sequential(
            nn.Conv1d(3,32,5,padding=2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32,64,5,padding=2), nn.ReLU(), nn.AdaptiveAvgPool1d(1), nn.Flatten())
    def forward(s, x): return s.f(x)      # (N,64) 임베딩
def finetune_and_predict(encoder):
    """인코더를 동결하고 소수 라벨로 분류기(선형 프로브)를 학습해 val 예측."""
    encoder.eval()
    with torch.no_grad(): Fs = encoder(Xs.to(device)); Fv = encoder(Xv.to(device))
    clf = nn.Sequential(nn.Linear(64,64), nn.ReLU(), nn.Linear(64,6)).to(device)
    opt = torch.optim.Adam(clf.parameters(), 1e-3); crit = nn.CrossEntropyLoss()
    dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Fs.cpu(), ys), batch_size=64, shuffle=True)
    for ep in range(100):
        clf.train()
        for xb,yb in dl: xb,yb = xb.to(device), yb.to(device); opt.zero_grad(); crit(clf(xb),yb).backward(); opt.step()
    clf.eval()
    with torch.no_grad(): return clf(Fv).argmax(1).cpu().numpy()

## SimCLR 자기지도 사전학습(미라벨)

In [ ]:
encoder = Encoder().to(device)
proj = nn.Sequential(nn.Linear(64,64), nn.ReLU(), nn.Linear(64,32)).to(device)   # SimCLR 투영헤드
def augment(x):
    x = x + 0.1*torch.randn_like(x)                                              # 지터
    x = x * (0.8 + 0.4*torch.rand(x.size(0),1,1,device=x.device))               # 스케일
    return torch.roll(x, int(np.random.randint(-15,16)), dims=2)                # 시간 이동
def ntxent(z1, z2, tau=0.5):                                                     # NT-Xent 대조손실
    z = F.normalize(torch.cat([z1,z2],0), dim=1); N = z1.size(0)
    sim = z @ z.T / tau; sim.fill_diagonal_(-9e15)
    t = torch.arange(2*N, device=z.device); t = (t + N) % (2*N)
    return F.cross_entropy(sim, t)
opt = torch.optim.Adam(list(encoder.parameters())+list(proj.parameters()), 1e-3)
dl = torch.utils.data.DataLoader(Xbig, batch_size=256, shuffle=True)
for ep in range(80):                                                            # 미라벨 대량으로 SSL 사전학습
    encoder.train()
    for xb in dl:
        xb = xb.to(device); opt.zero_grad()
        loss = ntxent(proj(encoder(augment(xb))), proj(encoder(augment(xb))))
        loss.backward(); opt.step()
print("SSL 사전학습 완료")

## 예측 → submission.csv

In [ ]:
pred = finetune_and_predict(encoder)
pd.DataFrame({"id": np.arange(len(pred)), "label": pred.astype(int)}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred))

증강 다양화·투영차원·에폭·프로브 미세조정으로 더 오를 수 있다.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)